In [1]:
import pandas as pd
from pathlib import Path

In [3]:
DROUGHT_CSV = Path("/Users/ramilmammadov/Desktop/Capstone Project/GitHub Repo/Capstone_Citizensbank_climate-risk_insurance-exposure_monitoring/02_processed_data/NCEI_annual_drought_county_FINAL.csv")
WILDFIRE_CSV = Path("/Users/ramilmammadov/Desktop/Capstone Project/GitHub Repo/Capstone_Citizensbank_climate-risk_insurance-exposure_monitoring/02_processed_data/wildfire_data_preprocessed.csv")
OUTPUT_CSV  = Path("/Users/ramilmammadov/Desktop/Capstone Project/GitHub Repo/Capstone_Citizensbank_climate-risk_insurance-exposure_monitoring/02_processed_data/merged_wildfire_drought.csv")

def standardize_keys(df):
    df = df.rename(columns={c: c.strip() for c in df.columns})
    lower_to_orig = {c.lower(): c for c in df.columns}
    ren = {}

    for k in ("fips", "fips_code", "geoid"):
        if k in lower_to_orig:
            ren[lower_to_orig[k]] = "FIPS"
            break

    for k in ("year", "fire_year"):
        if k in lower_to_orig:
            ren[lower_to_orig[k]] = "Year"
            break

    df = df.rename(columns=ren)

    if "FIPS" in df.columns:
        df["FIPS"] = df["FIPS"].astype(str).str.zfill(5)
    else:
        raise KeyError("FIPS column not found after standardization.")

    if "Year" in df.columns:
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    else:
        raise KeyError("Year column not found after standardization.")

    return df

drought = pd.read_csv(DROUGHT_CSV)
wildfire = pd.read_csv(WILDFIRE_CSV)

drought = standardize_keys(drought)
wildfire = standardize_keys(wildfire)

merged = pd.merge(
    drought,
    wildfire,
    on=["FIPS", "Year"],
    how="inner",                      
    suffixes=("_ncei", "_fire")     
)

merged.to_csv(OUTPUT_CSV, index=False)
print(f"Saved merged file to: {OUTPUT_CSV}")
print(f"Rows: {len(merged):,} | Columns: {len(merged.columns)}")

merged


Saved merged file to: /Users/ramilmammadov/Desktop/Capstone Project/GitHub Repo/Capstone_Citizensbank_climate-risk_insurance-exposure_monitoring/02_processed_data/merged_wildfire_drought.csv
Rows: 41,690 | Columns: 9


,FIPS,County,State_ncei,State_abbr,Year,Annual_Mean_Index,FIRE_FREQUENCY,TOTAL_FIRE_SIZE,State_fire
0,01001,Autauga County,Alabama,AL,2003,2.39,43,274.80,AL
1,01001,Autauga County,Alabama,AL,2004,0.00,86,744.50,AL
2,01001,Autauga County,Alabama,AL,2005,0.90,63,234.20,AL
3,01001,Autauga County,Alabama,AL,2006,-1.34,62,534.70,AL
4,01001,Autauga County,Alabama,AL,2007,-3.33,93,477.30,AL
...,...,...,...,...,...,...,...,...,...
41685,56045,Weston County,Wyoming,WY,2016,-2.27,34,2775.14,WY
41686,56045,Weston County,Wyoming,WY,2017,-1.32,33,96.40,WY
41687,56045,Weston County,Wyoming,WY,2018,4.41,15,96.90,WY
41688,56045,Weston County,Wyoming,WY,2019,7.09,13,93.05,WY
